## 1) Crawl dữ liệu bất động sản TP.HCM từ MoSo API
Thu thập dữ liệu tin đăng bất động sản (nhà/đất/căn hộ)
từ nền tảng MoSo API hoặc trang web tương ứng.

Mục tiêu:
- Lấy thông tin cơ bản: tiêu đề, giá, diện tích, địa chỉ, ngày đăng.
- Chuẩn hoá và lưu thành CSV để phục vụ tiền xử lý & huấn luyện mô hình.

Đầu ra:
- File: moso_api_data.csv
- Cấu trúc cột: [Tiêu đề, Giá, Diện tích sử dụng, Diện tích đất, Ngày đăng, Địa chỉ, ...]

## 2) Import & Cấu hình


In [8]:
import json
import csv
import time
import requests
import pandas as pd
from copy import deepcopy
from typing import Any, Dict, List

API_URL = "https://moso.vn/api"
REQUEST_JSON_PATH = "request.json"
OUTFILE = "moso_api_data.csv"

FIELDS = [
    "URL", "Giá", "Diện tích sử dụng", "Diện tích đất",
    "Phòng ngủ", "Phòng tắm", "Giấy tờ pháp lý", "Ngày đăng", "Địa chỉ"
]


## 3) HTTP Client (session)

In [9]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Origin": "https://moso.vn",
    "Referer": "https://moso.vn/",
    "X-Requested-With": "XMLHttpRequest",
})


## 4) Alias & Helpers


In [10]:
# Bảng ánh xạ tên trường khác nhau trong JSON (MOSO.vn có thể dùng nhiều tên cho cùng 1 dữ liệu)
ALIAS = {
    "price":  ["priceText", "price"],
    "usable": ["pArea", "usableArea"],
    "land":   ["pLandArea", "landArea"],
    "bed":    ["pNumberOfBedrooms", "bedrooms"],
    "bath":   ["pNumberOfBathrooms", "bathrooms"],
    "legal":  ["pCertificateType", "publicCertificate"],
    "date":   ["publishedAt", "_createdAt"],
}

# Gửi request POST đến API MOSO.vn và trả về dữ liệu JSON
def api_post(payload: Dict[str, Any]) -> Dict[str, Any]:
    resp = SESSION.post(API_URL, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

# Chuyển định dạng ngày từ ISO (yyyy-mm-dd) → dd/mm/yyyy
def format_date_iso_to_ddmmyyyy(iso_str: str) -> str:
    y, m, d = map(int, iso_str[:10].split("-"))
    return f"{d:02d}/{m:02d}/{y}"

# Định dạng giá tiền thành chuỗi dễ đọc (Triệu / Tỷ)
def format_price_display(v: Any) -> str:
    val = float(v)
    if val >= 1e9:
        return f"{val/1e9:.1f}".rstrip("0.") + " Tỷ"
    if val >= 1e6:
        return f"{val/1e6:.0f} Triệu"
    return str(val)

# Tạo URL chi tiết cho bài đăng (nếu chưa có thì ghép từ _id)
def build_item_url(item: Dict[str, Any]) -> str:
    page = item.get("@page") or item.get("page") or {}
    url = page.get("canonicalUrl") or page.get("url") or ""
    if not url and item.get("_id"):
        url = f"/ban-{item['_id']}"
    return ("https://moso.vn" + url) if (url and not url.startswith("http")) else (url or "")

# Kiểm tra giá trị có hợp lệ không (không None, không rỗng, không 0)
def is_valid_value(v: Any) -> bool:
    return v not in (None, "", 0, 0.0)

# Lấy giá trị đầu tiên có ý nghĩa (không rỗng) trong danh sách
def first_nonempty(values: List[Any]) -> Any:
    for v in values:
        if is_valid_value(v):
            return v
    return ""

# Tìm giá trị đầu tiên trong JSON theo danh sách các khóa (duyệt đệ quy)
def find_first_value(obj: Any, keys: List[str]) -> Any:
    found: List[Any] = []

    def walk(x: Any) -> None:
        if isinstance(x, dict):
            for k, v in x.items():
                if k in keys and is_valid_value(v):
                    found.append(v)
                elif isinstance(v, (dict, list)):
                    walk(v)
        elif isinstance(x, list):
            for v in x:
                walk(v)

    walk(obj)
    return first_nonempty(found)


## 5) Địa chỉ & build từng dòng dữ liệu


In [11]:
# Hàm lấy địa chỉ đầy đủ từ dữ liệu JSON tin đăng
def extract_address(item: Dict[str, Any]) -> str:
    # Hàm con: chuyển object địa chỉ (dict hoặc string) thành chuỗi địa chỉ đầy đủ
    def to_full(addr_obj: Any) -> str | None:
        if isinstance(addr_obj, dict):
            if addr_obj.get("full"):
                return str(addr_obj["full"])
            # Ghép các phần: tên đường, phường, quận, tỉnh/thành nếu có
            parts = [addr_obj.get(k) for k in ("streetName", "ward", "district", "provinceCity") if addr_obj.get(k)]
            if parts:
                return ", ".join(map(str, parts))
        elif isinstance(addr_obj, str):
            return addr_obj
        return None

    candidates: List[str] = []  # Danh sách các địa chỉ tìm được

    # Hàm con: duyệt toàn bộ JSON (có thể lồng dict/list)
    def walk(x: Any) -> None:
        if isinstance(x, dict):
            for k, v in x.items():
                # Tìm các khóa chứa địa chỉ
                if k in ("pAddress", "newPropertyAddress"):
                    s = to_full(v)
                    if s: candidates.append(s)
                elif k in ("_pAddress", "_newPropertyAddress", "pDetailedAddress", "address"):
                    s = to_full(v)
                    if s: candidates.append(s)
                # Tiếp tục duyệt sâu hơn nếu giá trị là dict/list
                if isinstance(v, (dict, list)):
                    walk(v)
        elif isinstance(x, list):
            for v in x:
                walk(v)

    walk(item)  # Bắt đầu duyệt JSON

    # Loại trùng lặp và trả về địa chỉ đầu tiên hợp lệ
    seen: set[str] = set()
    for s in candidates:
        ss = str(s).strip()
        if ss and ss not in seen:
            seen.add(ss)
            return ss
    return ""


# Hàm tạo 1 dòng dữ liệu hoàn chỉnh cho 1 tin đăng
def build_row(item: Dict[str, Any]) -> Dict[str, Any]:
    # Lấy thông tin pháp lý (giấy tờ)
    legal = find_first_value(item, ALIAS["legal"])
    if isinstance(legal, bool):
        legal = "Có giấy tờ" if legal else ""

    # Trả về 1 dòng dữ liệu dạng dict (sẽ ghi vào CSV)
    return {
        "URL": build_item_url(item),  # Link chi tiết bài đăng
        "Giá": format_price_display(find_first_value(item, ALIAS["price"])),  # Giá hiển thị
        "Diện tích sử dụng": find_first_value(item, ALIAS["usable"]),  # Diện tích sàn
        "Diện tích đất": find_first_value(item, ALIAS["land"]),  # Diện tích đất
        "Phòng ngủ": find_first_value(item, ALIAS["bed"]),  # Số phòng ngủ
        "Phòng tắm": find_first_value(item, ALIAS["bath"]),  # Số phòng tắm
        "Giấy tờ pháp lý": legal,  # Giấy tờ pháp lý
        "Ngày đăng": format_date_iso_to_ddmmyyyy(str(find_first_value(item, ALIAS["date"]))),  # Ngày đăng
        "Địa chỉ": extract_address(item),  # Địa chỉ chi tiết
    }


## 6) Hàm crawl & ghi CSV


In [12]:
# Hàm chính: Gửi request đến MOSO API, thu thập toàn bộ tin đăng và ghi vào file CSV
def crawl_and_write_csv() -> int:
    # Đọc payload mẫu (request.json) làm form gửi API
    with open(REQUEST_JSON_PATH, encoding="utf-8") as f:
        base = json.load(f)

    payload = deepcopy(base)  # Sao chép để không làm thay đổi file gốc
    limit = base.get("options", {}).get("limit", 100)  # Số lượng tin mỗi trang
    offset = 0  # Vị trí bắt đầu (phân trang)
    rows: List[Dict[str, Any]] = []  # Danh sách kết quả (các dòng dữ liệu)
    seen_urls: set[str] = set()  # Dùng để tránh trùng URL

    # Vòng lặp phân trang - lấy hết dữ liệu từng trang
    while True:
        payload.setdefault("options", {})
        payload["options"]["offset"] = offset  # Gán offset mới cho mỗi lượt request

        data = api_post(payload)  # Gửi request thật đến MOSO.vn
        models = data.get("models", [])  # Danh sách tin đăng trong trang hiện tại
        if not models:
            print(f"[{offset}] Không có dữ liệu, dừng.")
            break  # Hết dữ liệu thì dừng

        added = 0  # Đếm số tin mới thêm trong vòng lặp này
        for item in models:
            url = build_item_url(item)
            if not url or url in seen_urls:
                continue  # Bỏ qua nếu URL trống hoặc đã lấy rồi
            rows.append(build_row(item))  # Xử lý và thêm dòng dữ liệu
            seen_urls.add(url)
            added += 1

        print(f"[{offset}] +{added} (tổng {len(rows)})")  # Log tiến trình

        total = data.get("count") or 0  # Tổng số tin trên toàn hệ thống
        if added == 0:
            break  # Không có tin mới thì dừng
        if total and len(rows) >= total:
            break  # Đã lấy đủ tổng số tin thì dừng

        offset += limit  # Sang trang tiếp theo
        time.sleep(0.2)  # Nghỉ 0.2s để tránh bị chặn request

    # Ghi toàn bộ dữ liệu vào file CSV
    with open(OUTFILE, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(rows)

    print(f"[DONE] Ghi {len(rows)} dòng vào {OUTFILE}")

    return len(rows)  # Trả về số dòng đã ghi được


## 7) Thực thi khi sẵn sàng (`request.json` đã có)


In [13]:

total_rows = crawl_and_write_csv()
display("Số dòng thu thập:", total_rows)


[0] +100 (tổng 100)
[100] +100 (tổng 200)
[200] +100 (tổng 300)
[300] +100 (tổng 400)
[400] +100 (tổng 500)
[500] +100 (tổng 600)
[600] +100 (tổng 700)
[700] +100 (tổng 800)
[800] +100 (tổng 900)
[900] +100 (tổng 1000)
[1000] +100 (tổng 1100)
[1100] +100 (tổng 1200)
[1200] +100 (tổng 1300)
[1300] +100 (tổng 1400)
[1400] +100 (tổng 1500)
[1500] +100 (tổng 1600)
[1600] +100 (tổng 1700)
[1700] +100 (tổng 1800)
[1800] +100 (tổng 1900)
[1900] +100 (tổng 2000)
[2000] +100 (tổng 2100)
[2100] +100 (tổng 2200)
[2200] +30 (tổng 2230)
[DONE] Ghi 2230 dòng vào moso_api_data.csv


'Số dòng thu thập:'

2230

## 8) Xem nhanh CSV


In [14]:
try:
    df = pd.read_csv(OUTFILE)
    display(df.head(10))
except FileNotFoundError:
    print(f"Chưa thấy {OUTFILE}. Hãy chạy crawl trước.")


,URL,Giá,Diện tích sử dụng,Diện tích đất,Phòng ngủ,Phòng tắm,Giấy tờ pháp lý,Ngày đăng,Địa chỉ
0,https://moso.vn/ban-690388d2d638883988715f20,23.5 Tỷ,256.51,162.0,4.0,4.0,certificate,30/10/2025,"46 Đường Số 20, Hiệp Bình Chánh, Thủ Đức, Hồ C..."
1,https://moso.vn/ban-690375a5d638883988712dd5,4.8 Tỷ,NaN,79.8,NaN,NaN,certificate,30/10/2025,"Đường Lê Đức Thọ, 16, Gò Vấp, Hồ Chí Minh"
2,https://moso.vn/ban-69034b76d63888398870c2cf,4.8 Tỷ,60.60,77.1,6.0,6.0,certificate,30/10/2025,"2/23A Đường số 13, Linh Xuân, Thủ Đức, Hồ Chí ..."
3,https://moso.vn/ban-69034961d63888398870bc4c,21.5 Tỷ,66.20,66.2,2.0,1.0,certificate,30/10/2025,"801/43 Đường Xô Viết Nghệ Tĩnh, 25, Bình Thạnh..."
4,https://moso.vn/ban-6903475dd63888398870b555,1.8 Tỷ,26.00,9.3,1.0,1.0,certificate,30/10/2025,"40/13/27 Đường Số 2, 3, Gò Vấp, Hồ Chí Minh"
5,https://moso.vn/ban-69033ccbd63888398870990e,8.3 Tỷ,105.60,32.3,3.0,4.0,certificate,30/10/2025,"315/26A Đường Lê Văn Sỹ, 13, 3, Hồ Chí Minh"
6,https://moso.vn/ban-69033875d638883988708c97,5.5 Tỷ,111.90,231.6,1.0,2.0,certificate,30/10/2025,"D2/49 Quốc Lộ 50, Ấp 4, Đa Phước, Bình Chánh, ..."
7,https://moso.vn/ban-690332a6d638883988707bf0,8.5 Tỷ,123.30,45.7,4.0,3.0,certificate,30/10/2025,"524/16/19 Nguyễn Đình Chiểu, 4, 3, Hồ Chí Minh"
8,https://moso.vn/ban-6903322dd6388839887078b6,12.5 Tỷ,282.20,120.0,3.0,3.0,certificate,30/10/2025,"63 Đường Số 5, Khu nhà ở Bắc Lương Bèo, Tân Tạ..."
9,https://moso.vn/ban-69032f42d6388839887071cc,12.5 Tỷ,211.10,75.0,5.0,5.0,certificate,30/10/2025,"152/29 Bông Sao, 5, 8, Hồ Chí Minh"
